# Step 09 – Correlation, confusion-matrix and SHAP figures

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 3.1–3.4 · Figure 3, Figure 4, Table 3, Figure 5, Figure S2
- **Needs:** steps 03 and 04
- **Writes:** `RESULTS_DIR/figures_revision/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / figures_revision/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost shap

In [ ]:
"""
Figures from the re-run models

Run AFTER step 04 (04_model_tuning_and_evaluation) (uses the saved models, predictions and split indices).
In Colab first:  !pip install -q catboost shap

Part A  Confusion matrices of all ten classifiers, 2 x 5 panels, shared colour bar,
                row labels only on the left panels (fixes the old 10th-panel layout)
Part B  Figure 3: Pearson correlation matrices on the TRAINING partition only
                  - the 16 retained predictors (main text)
                  - all candidate predictors, retained ones marked (supplementary)
Part C  CatBoost SHAP values on the 840 TEST points
                  (CatBoost's exact built-in TreeSHAP, one value per class)
                  - global importance: mean |SHAP| per predictor, stacked by class
                  - beeswarm for the forest class
                  - dependence of the forest-class SHAP on slope and elevation

Outputs (OUT_DIR): Fig_confusion_all10, Fig3_correlation_retained, FigS_correlation_all,
                   Fig_shap_global, Fig_shap_beeswarm_forest, Fig_shap_terrain_forest
                   (PNG 600 dpi + PDF), shap_mean_abs_by_class.csv, correlation_retained.csv
"""


import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import confusion_matrix
from catboost import Pool
import shap

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'
OUT_DIR    = RESULTS_DIR + 'figures_revision/'
LABEL_COL  = 'class'
CLASS_NAMES  = {0: 'Water', 1: 'Forest', 2: 'Bare ground', 3: 'Urban', 4: 'Crops/grass'}
CLASS_COLORS = {'Water': '#2C7FB8', 'Forest': '#1B7837', 'Bare ground': '#C9A27E',
                'Urban': '#D7301F', 'Crops/grass': '#E6C229'}
FOREST_CODE = 1
MODELS = ['CatBoost', 'XGBoost', 'MLP', 'GTB', 'RF', 'SVM-RBF', 'KNN', 'CART', 'MDC', 'NB']
NOT_PREDICTORS = {LABEL_COL, 'longitude', 'latitude'}   # numeric columns that are not candidates

os.makedirs(OUT_DIR, exist_ok=True)
ORDER = list(CLASS_NAMES.values())

def save(fig, name):
    fig.savefig(OUT_DIR + name + '.png', dpi=600, bbox_inches='tight')
    fig.savefig(OUT_DIR + name + '.pdf', bbox_inches='tight')
    plt.show()

PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()   # model order
df = pd.read_csv(DATA_PATH)
train_idx = pd.read_csv(MODEL_DIR + 'train_indices.csv')['train_idx'].values
test_idx  = pd.read_csv(MODEL_DIR + 'test_indices.csv')['test_idx'].values
pred = pd.read_csv(MODEL_DIR + 'test_predictions.csv')
summary = pd.read_csv(MODEL_DIR + 'train_cv_test_full.csv').set_index('Model')
assert len(train_idx) == 1960 and len(test_idx) == 840

# Display order for predictors: bands, indices, texture 7x7, texture 11x11, terrain
def group(f):
    if f in ('slope', 'elevation'):  return 4
    if f.endswith('_11x11'):         return 3
    if f.endswith('_7x7'):           return 2
    if f.startswith('B') and f[1:].replace('A', '').isdigit(): return 0
    return 1
DISPLAY = sorted(PREDICTORS, key=lambda f: (group(f), f))

# ================================================================== A. confusion matrices
yt = pred['y_true'].map(CLASS_NAMES)
fig, axes = plt.subplots(2, 5, figsize=(7.1, 3.45), sharex=True, sharey=True)
for ax, m in zip(axes.flat, MODELS):
    cm = confusion_matrix(yt, pred[m].map(CLASS_NAMES), labels=ORDER)
    pct = 100 * cm / cm.sum(1, keepdims=True)
    im = ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(len(ORDER)):
        for j in range(len(ORDER)):
            if cm[i, j]:
                ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=5.5,
                        color='white' if pct[i, j] > 60 else 'black')
    ax.set_title(f"{m} ({100 * summary.loc[m, 'Test_OA']:.2f}%)", fontsize=7, fontweight='bold', pad=3)
    ax.set_xticks(range(len(ORDER)), ORDER, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(len(ORDER)), ORDER, fontsize=6)
    ax.tick_params(length=2, pad=1)
fig.supxlabel('Predicted class', fontsize=8, y=0.02)
fig.supylabel('Reference class', fontsize=8, x=0.0)
fig.subplots_adjust(left=0.125, right=0.9, bottom=0.21, top=0.93, wspace=0.08, hspace=0.12)
cax = fig.add_axes([0.915, 0.24, 0.012, 0.66])
cb = fig.colorbar(im, cax=cax)
cb.set_label('% of reference class', fontsize=7)
cb.ax.tick_params(labelsize=6)
save(fig, 'Fig_confusion_all10')
print('A. Confusion matrices: done (numbers = test points; shading = % of the reference class)')

# ================================================================== B. correlation matrices
def corr_heatmap(cols, name, annotate, retained=None, size=(6.5, 5.6)):
    c = df.loc[train_idx, cols].corr(method='pearson')
    fig, ax = plt.subplots(figsize=size)
    mask = np.triu(np.ones_like(c, dtype=bool), k=1)
    im = ax.imshow(np.ma.masked_array(c.values, mask), cmap='RdBu_r', norm=TwoSlopeNorm(0, -1, 1))
    if annotate:
        for i in range(len(cols)):
            for j in range(i + 1):
                ax.text(j, i, f'{c.iat[i, j]:.2f}', ha='center', va='center', fontsize=5,
                        color='white' if abs(c.iat[i, j]) > 0.6 else 'black')
    fs = 6 if len(cols) <= 20 else 4.5
    ax.set_xticks(range(len(cols)), cols, rotation=90, fontsize=fs)
    ax.set_yticks(range(len(cols)), cols, fontsize=fs)
    if retained is not None:
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            if lab.get_text() in retained:
                lab.set_fontweight('bold')
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, shrink=0.7)
    cb.set_label('Pearson r (training partition, n = 1,960)', fontsize=7)
    cb.ax.tick_params(labelsize=6)
    fig.tight_layout()
    save(fig, name)
    return c

c16 = corr_heatmap(DISPLAY, 'Fig3_correlation_retained', annotate=True)
c16.round(3).to_csv(OUT_DIR + 'correlation_retained.csv')
off = c16.where(~np.eye(len(c16), dtype=bool)).abs()
print(f'B. Retained predictors: max |r| = {off.max().max():.3f} (should be <= 0.80)')

candidates = [c for c in df.select_dtypes('number').columns if c not in NOT_PREDICTORS]
print(f'   Candidate predictors found in the CSV: {len(candidates)} (expected 56)')
cand_sorted = sorted(candidates, key=lambda f: (group(f), f))
corr_heatmap(cand_sorted, 'FigS_correlation_all', annotate=False, retained=set(PREDICTORS), size=(8, 7.2))
print('   Supplementary: all candidates, retained predictors in bold')

# ================================================================== C. SHAP for CatBoost
cat = joblib.load(MODEL_DIR + 'CatBoost.joblib')
X_test = df.loc[test_idx, PREDICTORS]
sv = cat.get_feature_importance(Pool(X_test), type='ShapValues')    # (n, classes, features + 1)
assert sv.ndim == 3 and sv.shape[1] == len(CLASS_NAMES), f'Unexpected SHAP shape {sv.shape}'
sv = sv[:, :, :-1]                                                   # drop the expected-value column
classes = [CLASS_NAMES[int(c)] for c in cat.classes_]
fi = classes.index('Forest')

mean_abs = pd.DataFrame(np.abs(sv).mean(0).T, index=PREDICTORS, columns=classes)
mean_abs = mean_abs.assign(Total=mean_abs.sum(1))
mean_abs = mean_abs.sort_values('Total', ascending=False)
mean_abs.round(4).to_csv(OUT_DIR + 'shap_mean_abs_by_class.csv')
print('\nC. Mean |SHAP| (test set), top predictors:\n', mean_abs.round(3).head(10).to_string())

# C1 global importance, stacked by class
fig, ax = plt.subplots(figsize=(5.2, 4.2))
order = mean_abs.index[::-1]
left = np.zeros(len(order))
for cl in classes:
    ax.barh(order, mean_abs.loc[order, cl], left=left, color=CLASS_COLORS[cl], label=cl, height=0.7)
    left += mean_abs.loc[order, cl].values
ax.set_xlabel('Mean |SHAP value| (test set, n = 840)', fontsize=8)
ax.tick_params(labelsize=7)
ax.legend(fontsize=7, frameon=False, loc='lower right')
ax.grid(axis='x', alpha=0.3, lw=0.5)
fig.tight_layout()
save(fig, 'Fig_shap_global')

# C2 beeswarm, forest class
plt.figure()
shap.summary_plot(sv[:, fi, :], X_test, feature_names=PREDICTORS, show=False, plot_size=(5.5, 4.5),
                  max_display=len(PREDICTORS))
fig = plt.gcf()
fig.axes[0].set_xlabel('SHAP value for the forest class (log-odds)', fontsize=8)
save(fig, 'Fig_shap_beeswarm_forest')

# C3 terrain dependence, forest class
true_cls = pred.set_index('row_idx').loc[test_idx, 'y_true'].map(CLASS_NAMES).values
fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.0), sharey=True)
for ax, f in zip(axes, ['slope', 'elevation']):
    j = PREDICTORS.index(f)
    for cl in ORDER:
        sel = true_cls == cl
        ax.scatter(X_test[f].values[sel], sv[sel, fi, j], s=6, alpha=0.7,
                   color=CLASS_COLORS[cl], label=cl, edgecolor='none')
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.set_xlabel({'slope': 'Slope (°)', 'elevation': 'Elevation (m)'}[f], fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3, lw=0.5)
axes[0].set_ylabel('SHAP value for forest (log-odds)', fontsize=8)
axes[1].legend(fontsize=6.5, frameon=False, markerscale=2, title='Reference class', title_fontsize=6.5)
fig.tight_layout()
save(fig, 'Fig_shap_terrain_forest')

print('\nFiles saved to:', OUT_DIR)